In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import joblib
import os


In [2]:
# load data set 

df = pd.read_csv('../data./raw/upi_fraud_25.csv')

In [3]:
df.head(4)

,Transaction ID,Timestamp,Sender ID,Receiver ID,Amount,Merchant category,Transaction type,Location,Device type,UPI channel,Hour of day,Day of week,Historical spend pattern,Risk label,Anomaly flag,Amount vs Historical Spend,Risk score
0,TXN0014696,2025-01-01 00:02:46,S00615,R06267,377.65,Online Services,P2P,Delhi,Android,Online,0,Wednesday,134.14,Genuine,0,2.815,36.89
1,TXN0023230,2025-01-01 00:05:29,S00728,R07201,111.95,Utilities,Online Purchase,Nagpur,Android,Online,0,Wednesday,139.68,Genuine,0,0.801,24.81
2,TXN0009755,2025-01-01 00:09:12,S00611,R05418,214.38,Healthcare,P2P,Bengaluru,Android,QR,0,Wednesday,284.03,Genuine,0,0.755,24.53
3,TXN0008446,2025-01-01 03:42:46,S04644,R01103,413.26,Healthcare,Online Purchase,Bhopal,Android,Mobile Number,3,Wednesday,85.30,Fraud,1,4.845,69.07


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Transaction ID              25000 non-null  str    
 1   Timestamp                   25000 non-null  str    
 2   Sender ID                   25000 non-null  str    
 3   Receiver ID                 25000 non-null  str    
 4   Amount                      25000 non-null  float64
 5   Merchant category           25000 non-null  str    
 6   Transaction type            25000 non-null  str    
 7   Location                    25000 non-null  str    
 8   Device type                 25000 non-null  str    
 9   UPI channel                 25000 non-null  str    
 10  Hour of day                 25000 non-null  int64  
 11  Day of week                 25000 non-null  str    
 12  Historical spend pattern    25000 non-null  float64
 13  Risk label                  25000 non-null

In [5]:
df.shape

(25000, 17)

In [6]:
# Remove duplicate transaction IDs
df = df.drop_duplicates(
    subset=["Transaction ID"],
    keep="first"
).copy()

In [7]:
# Convert data types
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    errors="coerce"
)

In [8]:
df["Amount"] = pd.to_numeric(
    df["Amount"],
    errors="coerce"
)

In [9]:
# df.info()

In [10]:
# Remove rows with invalid timestamp or amount
df = df.dropna(
    subset=["Timestamp", "Amount"]
).copy()


Step 4 — Create time-series features

In [11]:
df.head(2)

,Transaction ID,Timestamp,Sender ID,Receiver ID,Amount,Merchant category,Transaction type,Location,Device type,UPI channel,Hour of day,Day of week,Historical spend pattern,Risk label,Anomaly flag,Amount vs Historical Spend,Risk score
0,TXN0014696,2025-01-01 00:02:46,S00615,R06267,377.65,Online Services,P2P,Delhi,Android,Online,0,Wednesday,134.14,Genuine,0,2.815,36.89
1,TXN0023230,2025-01-01 00:05:29,S00728,R07201,111.95,Utilities,Online Purchase,Nagpur,Android,Online,0,Wednesday,139.68,Genuine,0,0.801,24.81


In [12]:
df = df.sort_values("Timestamp").reset_index(drop=True)

# Extract time information
df["Hour"] = df["Timestamp"].dt.hour

df["Day of Week"] = df["Timestamp"].dt.dayofweek

df["Day of Month"] = df["Timestamp"].dt.day

df["Month"] = df["Timestamp"].dt.month

# Monday = 0, Sunday = 6
df["Is Weekend"] = (
    df["Day of Week"] >= 5
).astype(int)

# Transactions from midnight to 4:59 AM
df["Is Night"] = (
    df["Hour"] < 5
).astype(int)

display(
    df[
        [
            "Timestamp",
            "Hour",
            "Day of Week",
            "Day of Month",
            "Month",
            "Is Weekend",
            "Is Night"
        ]
    ].head()
)

,Timestamp,Hour,Day of Week,Day of Month,Month,Is Weekend,Is Night
0,2025-01-01 00:02:46,0,2,1,1,0,1
1,2025-01-01 00:05:29,0,2,1,1,0,1
2,2025-01-01 00:09:12,0,2,1,1,0,1
3,2025-01-01 00:29:40,0,2,1,1,0,1
4,2025-01-01 00:31:28,0,2,1,1,0,1


Step 5 — Create transaction timing features:-

        5.1 Time since the sender's previous transaction

In [13]:
# Sort by sender and timestamp
df = df.sort_values(
    ["Sender ID", "Timestamp"]
).reset_index(drop=True)

# Time difference between consecutive transactions
df["Time Since Previous Txn (minutes)"] = (
    df.groupby("Sender ID")["Timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

display(
    df[
        [
            "Sender ID",
            "Timestamp",
            "Time Since Previous Txn (minutes)"
        ]
    ].head(10)
)

,Sender ID,Timestamp,Time Since Previous Txn (minutes)
0,S00001,2025-01-09 20:24:41,NaN
1,S00001,2025-01-15 03:16:32,7611.850000
2,S00001,2025-03-23 01:51:21,96394.816667
3,S00001,2025-04-24 04:19:19,46227.966667
4,S00001,2025-05-01 16:04:48,10785.483333
5,S00001,2025-06-02 23:19:52,46515.066667
6,S00002,2025-01-03 02:27:08,NaN
7,S00002,2025-04-16 09:58:58,148771.833333
8,S00002,2025-06-16 15:29:23,88170.416667
9,S00002,2025-06-28 13:10:42,17141.316667


5.2 Count previous transactions in short time windows

In [14]:
# Preserve a unique row identifier
df["_row_id"] = np.arange(len(df))  # id from 0 to one less of total no of rows, 0, 24999

# Initialize the features
history_features = [
    "Previous 5m Txn Count",
    "Previous 5m Total Amount",
    "Previous Hour Txn Count",
    "Previous Hour Total Amount",
    "Previous 24h Txn Count",
    "Previous 24h Total Amount"
]

for col in history_features:
    df[col] = 0.0

# Process each sender separately
for sender, group in df.groupby("Sender ID", sort=False):

    group = group.sort_values("Timestamp")

    # Use timestamps as the index for rolling calculations
    time_data = group.set_index("Timestamp")["Amount"]

    # Count transactions strictly before the current timestamp
    count_5m = time_data.rolling(
        "5min", closed="left"
    ).count()

    amount_5m = time_data.rolling(
        "5min", closed="left"
    ).sum()

    count_1h = time_data.rolling(
        "1h", closed="left"
    ).count()

    amount_1h = time_data.rolling(
        "1h", closed="left"
    ).sum()

    count_24h = time_data.rolling(
        "24h", closed="left"
    ).count()

    amount_24h = time_data.rolling(
        "24h", closed="left"
    ).sum()

    # Assign results back to the correct rows
    row_ids = group["_row_id"].to_numpy()

    df.loc[row_ids, "Previous 5m Txn Count"] = (
        count_5m.to_numpy()
    )

    df.loc[row_ids, "Previous 5m Total Amount"] = (
        amount_5m.to_numpy()
    )

    df.loc[row_ids, "Previous Hour Txn Count"] = (
        count_1h.to_numpy()
    )

    df.loc[row_ids, "Previous Hour Total Amount"] = (
        amount_1h.to_numpy()
    )

    df.loc[row_ids, "Previous 24h Txn Count"] = (
        count_24h.to_numpy()
    )

    df.loc[row_ids, "Previous 24h Total Amount"] = (
        amount_24h.to_numpy()
    )

print(df[history_features].describe())

print(
    "\nTransactions with previous transactions in 24h:",
    (df["Previous 24h Txn Count"] > 0).sum()
)

       Previous 5m Txn Count  Previous 5m Total Amount  \
count                    4.0                  4.000000   
mean                     1.0                743.345000   
std                      0.0               1144.216101   
min                      1.0                 53.650000   
25%                      1.0                 77.192500   
50%                      1.0                237.615000   
75%                      1.0                903.767500   
max                      1.0               2444.500000   

       Previous Hour Txn Count  Previous Hour Total Amount  \
count                     19.0                   19.000000   
mean                       1.0                  762.873158   
std                        0.0                  763.652370   
min                        1.0                   12.680000   
25%                        1.0                  160.140000   
50%                        1.0                  436.590000   
75%                        1.0             

In [15]:
df.head(4)

,Transaction ID,Timestamp,Sender ID,Receiver ID,Amount,Merchant category,Transaction type,Location,Device type,UPI channel,...,Is Weekend,Is Night,Time Since Previous Txn (minutes),_row_id,Previous 5m Txn Count,Previous 5m Total Amount,Previous Hour Txn Count,Previous Hour Total Amount,Previous 24h Txn Count,Previous 24h Total Amount
0,TXN0019233,2025-01-09 20:24:41,S00001,R07877,110.78,Travel,P2M,Bengaluru,Android,QR,...,0,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
1,TXN0016071,2025-01-15 03:16:32,S00001,R01844,2091.40,Travel,P2P,Pune,Web,Online,...,0,1,7611.850000,1,NaN,NaN,NaN,NaN,NaN,NaN
2,TXN0024197,2025-03-23 01:51:21,S00001,R00617,647.40,Education,Recharge,Indore,iOS,Collect Request,...,1,1,96394.816667,2,NaN,NaN,NaN,NaN,NaN,NaN
3,TXN0015433,2025-04-24 04:19:19,S00001,R05258,51.94,Restaurants,P2M,Bengaluru,iOS,Mobile Number,...,0,1,46227.966667,3,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# df.tail(5)

Step 6 — Compare current spending with historical spending

In [17]:
# Previous transaction amount for each sender
df["Previous Txn Amount"] = (
    df.groupby("Sender ID")["Amount"].shift(1)
)

# Historical average using previous transactions only
df["Historical Amount Mean"] = (
    df.groupby("Sender ID")["Amount"]
    .transform(
        lambda s: s.shift(1).expanding().mean()
    )
)

# Historical standard deviation using previous transactions
df["Historical Amount Std"] = (
    df.groupby("Sender ID")["Amount"]
    .transform(
        lambda s: s.shift(1).expanding().std()
    )
)

# Difference between current amount and historical average
df["Amount Difference"] = (
    df["Amount"] - df["Historical Amount Mean"]
)

# Calculate a historical deviation score
df["Amount Deviation Score"] = (
    df["Amount Difference"]
    / df["Historical Amount Std"].replace(0, np.nan)
)

display(
    df[
        [
            "Sender ID",
            "Amount",
            "Historical Amount Mean",
            "Historical Amount Std",
            "Amount Difference",
            "Amount Deviation Score"
        ]
    ].head(15)
)

,Sender ID,Amount,Historical Amount Mean,Historical Amount Std,Amount Difference,Amount Deviation Score
0,S00001,110.78,NaN,NaN,NaN,NaN
1,S00001,2091.40,110.7800,NaN,1980.6200,NaN
2,S00001,647.40,1101.0900,1400.509833,-453.6900,-0.323946
3,S00001,51.94,949.8600,1024.365870,-897.9200,-0.876562
4,S00001,298.13,725.3800,949.270968,-427.2500,-0.450082
5,S00001,1545.12,639.9300,844.005356,905.1900,1.072493
6,S00002,839.37,NaN,NaN,NaN,NaN
7,S00002,200.41,839.3700,NaN,-638.9600,NaN
8,S00002,383.09,519.8900,451.812949,-136.8000,-0.302780
9,S00002,325.40,474.2900,329.098086,-148.8900,-0.452418


In [18]:
# 1. Flag whether the sender has a previous transaction
df["Has Previous Txn"] = (
    df["Time Since Previous Txn (minutes)"].notna()
).astype(int)

# 2. Flag whether a historical spending mean exists
df["Has Historical Mean"] = (
    df["Historical Amount Mean"].notna()
).astype(int)

# 3. Fill time gap where no previous transaction exists
df["Time Since Previous Txn (minutes)"] = (
    df["Time Since Previous Txn (minutes)"]
    .fillna(-1)
)

# 4. Fill previous transaction amount
df["Previous Txn Amount"] = (
    df["Previous Txn Amount"].fillna(0)
)

# 5. Fill historical mean and standard deviation
df["Historical Amount Mean"] = (
    df["Historical Amount Mean"].fillna(0)
)

df["Historical Amount Std"] = (
    df["Historical Amount Std"].fillna(0)
)

# 6. Fill amount difference
df["Amount Difference"] = (
    df["Amount Difference"].fillna(0)
)

# 7. Fill deviation score
df["Amount Deviation Score"] = (
    df["Amount Deviation Score"].fillna(0)
)

# Check the result
columns_to_check = [
    "Time Since Previous Txn (minutes)",
    "Previous Txn Amount",
    "Historical Amount Mean",
    "Historical Amount Std",
    "Amount Difference",
    "Amount Deviation Score",
    "Has Previous Txn",
    "Has Historical Mean"
]

print(df[columns_to_check].isnull().sum())

Time Since Previous Txn (minutes)    0
Previous Txn Amount                  0
Historical Amount Mean               0
Historical Amount Std                0
Amount Difference                    0
Amount Deviation Score               0
Has Previous Txn                     0
Has Historical Mean                  0
dtype: int64


Step 7 — Detect unusual amounts using the IQR method

In [19]:
Q1 = df["Amount"].quantile(0.25)
Q3 = df["Amount"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

df["IQR Anomaly"] = (
    (df["Amount"] < lower_limit) |
    (df["Amount"] > upper_limit)
).astype(int)

print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)

print("\nIQR anomaly counts:")
print(df["IQR Anomaly"].value_counts())

Lower limit: -1116.89375
Upper limit: 2174.67625

IQR anomaly counts:
IQR Anomaly
0    23266
1     1734
Name: count, dtype: int64


Step 8 — Detect hourly transaction spikes

In [20]:
# Count all transactions per hour
hourly = (
    df.set_index("Timestamp")
    .resample("h")
    .size()
    .rename("Hourly Txn Count")
    .to_frame()
)

# Baseline calculated only from previous hours
hourly["Previous 24h Mean"] = (
    hourly["Hourly Txn Count"]
    .rolling(window=24, min_periods=24)
    .mean()
    .shift(1)
)

hourly["Previous 24h Std"] = (
    hourly["Hourly Txn Count"]
    .rolling(window=24, min_periods=24)
    .std()
    .shift(1)
)

# Flag unusually high hourly transaction volume
hourly["Time Series Anomaly"] = (
    (
        hourly["Hourly Txn Count"]
        > hourly["Previous 24h Mean"]
        + 3 * hourly["Previous 24h Std"]
    )
    & (hourly["Hourly Txn Count"] >= 2)
    & hourly["Previous 24h Mean"].notna()
).astype(int)

# Attach hourly results to each transaction
df["Transaction Hour"] = df["Timestamp"].dt.floor("h")

df = df.merge(
    hourly[
        [
            "Hourly Txn Count",
            "Previous 24h Mean",
            "Previous 24h Std",
            "Time Series Anomaly"
        ]
    ],
    left_on="Transaction Hour",
    right_index=True,
    how="left",
    validate="many_to_one"
)

print("Hourly anomaly counts:")
print(df["Time Series Anomaly"].value_counts())

display(
    df[
        [
            "Timestamp",
            "Hourly Txn Count",
            "Previous 24h Mean",
            "Previous 24h Std",
            "Time Series Anomaly"
        ]
    ].head(10)
)

Hourly anomaly counts:
Time Series Anomaly
0    23186
1     1814
Name: count, dtype: int64


,Timestamp,Hourly Txn Count,Previous 24h Mean,Previous 24h Std,Time Series Anomaly
0,2025-01-09 20:24:41,1,2.791667,2.021174,0
1,2025-01-15 03:16:32,7,3.333333,2.792329,0
2,2025-03-23 01:51:21,9,2.208333,2.245366,1
3,2025-04-24 04:19:19,2,2.583333,1.998188,0
4,2025-05-01 16:04:48,1,2.958333,2.970568,0
5,2025-06-02 23:19:52,8,2.916667,2.780627,0
6,2025-01-03 02:27:08,6,2.291667,2.074256,0
7,2025-04-16 09:58:58,5,3.125000,3.138990,0
8,2025-06-16 15:29:23,1,2.875000,2.923852,0
9,2025-06-28 13:10:42,5,2.958333,3.113493,0


In [ ]:
# print(df.columns.tolist())

['Transaction ID', 'Timestamp', 'Sender ID', 'Receiver ID', 'Amount', 'Merchant category', 'Transaction type', 'Location', 'Device type', 'UPI channel', 'Hour of day', 'Day of week', 'Historical spend pattern', 'Risk label', 'Anomaly flag', 'Amount vs Historical Spend', 'Risk score', 'Hour', 'Day of Week', 'Day of Month', 'Month', 'Is Weekend', 'Is Night', 'Time Since Previous Txn (minutes)', '_row_id', 'Previous 5m Txn Count', 'Previous 5m Total Amount', 'Previous Hour Txn Count', 'Previous Hour Total Amount', 'Previous 24h Txn Count', 'Previous 24h Total Amount', 'Previous Txn Amount', 'Historical Amount Mean', 'Historical Amount Std', 'Amount Difference', 'Amount Deviation Score', 'Has Previous Txn', 'Has Historical Mean', 'IQR Anomaly', 'Transaction Hour', 'Hourly Txn Count', 'Previous 24h Mean', 'Previous 24h Std', 'Time Series Anomaly']


Step 9 — Prepare features for Isolation Forest

In [23]:
numeric_features = [
    "Amount",
    "Hour",
    "Day of Week",
    "Day of Month",
    "Month",
    "Is Weekend",
    "Is Night",
    "Time Since Previous Txn (minutes)",
    "Previous 5m Txn Count",
    "Previous 5m Total Amount",
    "Previous Hour Txn Count",
    "Previous Hour Total Amount",
    "Previous 24h Txn Count",
    "Previous 24h Total Amount",
    "Historical Amount Mean",
    "Amount Deviation Score",
    "Hourly Txn Count",
    "Historical spend pattern"
]

categorical_features = [
    "Merchant category",
    "Transaction type",
    "Location",
    "Device type",
    "UPI channel",
]

# Select model input features
X = df[numeric_features + categorical_features].copy()

# Convert infinite values to missing values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert categories into numerical indicator columns
X = pd.get_dummies(
    X,
    columns=categorical_features,
    dtype=int
)

# Remove columns that contain only missing values
all_missing_columns = X.columns[X.isna().all()]
X = X.drop(columns=all_missing_columns)

# Fill remaining missing values using each column's median
imputer = SimpleImputer(strategy="median")

X_imputed = imputer.fit_transform(X)

print("Number of model features:", X.shape[1])
print("Model input shape:", X_imputed.shape)
print("Any missing values:", np.isnan(X_imputed).any())

Number of model features: 58
Model input shape: (25000, 58)
Any missing values: False


Step 10 — Train the Isolation Forest model

In [24]:
model = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42,
    n_jobs=-1
)

# Train the unsupervised model
model.fit(X_imputed)

# Isolation Forest predicts:
#  1 = normal
# -1 = anomaly
predictions = model.predict(X_imputed)

# Convert results to convenient 0/1 labels
df["IF Anomaly"] = (
    predictions == -1
).astype(int)

# Higher score means more unusual
df["IF Anomaly Score"] = -model.score_samples(
    X_imputed
)

print("Isolation Forest results:")
print(df["IF Anomaly"].value_counts())

display(
    df[
        [
            "Transaction ID",
            "Amount",
            "IF Anomaly",
            "IF Anomaly Score"
        ]
    ].head(10)
)

Isolation Forest results:
IF Anomaly
0    24500
1      500
Name: count, dtype: int64


,Transaction ID,Amount,IF Anomaly,IF Anomaly Score
0,TXN0019233,110.78,0,0.416076
1,TXN0016071,2091.40,0,0.449247
2,TXN0024197,647.40,0,0.466916
3,TXN0015433,51.94,0,0.441567
4,TXN0012318,298.13,0,0.424714
5,TXN0011941,1545.12,0,0.447951
6,TXN0021504,839.37,0,0.436824
7,TXN0015493,200.41,0,0.432525
8,TXN0023022,383.09,0,0.431820
9,TXN0018948,325.40,0,0.444601


Step 11 — Combine the anomaly signals

    We'll combine three signals:
        1. IQR amount anomaly.
        2. Isolation Forest anomaly.
        3. Hourly time-series anomaly.

In [25]:
df["Anomaly Signal Count"] = (
    df["IQR Anomaly"]
    + df["IF Anomaly"]
    + df["Time Series Anomaly"]
)

# Flag transactions that trigger at least two signals
df["Fraud Review Flag"] = (
    df["Anomaly Signal Count"] >= 2
).astype(int)

# Heuristic risk score from 0 to 100
df["Combined Risk Score"] = (
    df["IQR Anomaly"] * 30
    + df["IF Anomaly"] * 40
    + df["Time Series Anomaly"] * 30
)

print("Review flag counts:")
print(df["Fraud Review Flag"].value_counts())

display(
    df.sort_values(
        "Combined Risk Score",
        ascending=False
    )[
        [
            "Transaction ID",
            "Timestamp",
            "Sender ID",
            "Amount",
            "IQR Anomaly",
            "IF Anomaly",
            "Time Series Anomaly",
            "Anomaly Signal Count",
            "Combined Risk Score",
            "Fraud Review Flag"
        ]
    ].head(20)
)

Review flag counts:
Fraud Review Flag
0    24492
1      508
Name: count, dtype: int64


,Transaction ID,Timestamp,Sender ID,Amount,IQR Anomaly,IF Anomaly,Time Series Anomaly,Anomaly Signal Count,Combined Risk Score,Fraud Review Flag
2961,TXN0019918,2025-11-12 00:49:48,S00585,2568.70,1,1,1,3,100,1
23888,TXN0015850,2025-09-03 03:29:14,S04775,4551.96,1,1,1,3,100,1
23314,TXN0006427,2025-04-01 22:02:15,S04668,10842.17,1,1,1,3,100,1
23085,TXN0004940,2025-01-06 00:40:16,S04622,2313.65,1,1,1,3,100,1
3628,TXN0018788,2025-10-31 22:25:43,S00717,2272.85,1,1,1,3,100,1
11749,TXN0024368,2025-02-20 22:08:20,S02338,7659.39,1,1,1,3,100,1
9582,TXN0006503,2025-09-08 03:26:20,S01909,12212.36,1,1,1,3,100,1
2908,TXN0018800,2025-08-30 02:12:48,S00571,2989.26,1,1,1,3,100,1
14459,TXN0016936,2025-04-04 22:12:55,S02887,3381.93,1,1,1,3,100,1
14411,TXN0007073,2025-12-25 02:17:04,S02875,7505.62,1,1,1,3,100,1


Step 12 — Check the final results

In [26]:
print("Final dataset shape:", df.shape)

print("\nMissing values in key features:")
print(
    df[
        [
            "Time Since Previous Txn (minutes)",
            "Historical Amount Mean",
            "Amount Deviation Score",
            "IF Anomaly Score"
        ]
    ].isnull().sum()
)

print("\nFraud review rate:")
print(
    round(df["Fraud Review Flag"].mean() * 100, 2),
    "%"
)

Final dataset shape: (25000, 49)

Missing values in key features:
Time Since Previous Txn (minutes)    0
Historical Amount Mean               0
Amount Deviation Score               0
IF Anomaly Score                     0
dtype: int64

Fraud review rate:
2.03 %


Step 13 — Save the processed dataset and model

In [27]:
# Save all transactions with features and anomaly results
df.drop(
    columns=["_row_id", "Transaction Hour"],
    errors="ignore"
).to_csv(
    "../data./processed/upi_fraud_processed.csv",
    index=False
)

# Save transactions flagged for review
df[df["Fraud Review Flag"] == 1].to_csv(
    "../data./processed/suspicious_transactions.csv",
    index=False
)

# Save hourly time-series results
hourly.to_csv(
    "../data./processed/hourly_anomaly_analysis.csv"
)

# Save model, imputer and feature metadata together
joblib.dump(
    {
        "model": model,
        "imputer": imputer,
        "feature_columns": X.columns.tolist(),
        "all_missing_columns": all_missing_columns.tolist(),
        "contamination": 0.02
    },
    "../models/upi_isolation_forest_bundle.pkl"
)

print("All files saved successfully.")

All files saved successfully.
